# Physical Lab and Data

Gymnasium environments standardize code implementation of an RL problem. To create an environment working with real hardware, such as a robot arm, one will need to call corresponding APIs inside interfaces like `step()` and `get_obs()`. DIKU Lab space is equipped with various robots and sensors to research ideas and applications. Prior to using the lab space, one should attend and pass training at [an Absalon course](https://absalon.ku.dk/enroll/RFHRPY). A [website](https://diku-dk.github.io/image-website/robotlab/) is maintained to provide basic introduction and pointers on using these hardware. This course note module will not cover these but focus on optical tracking system, for retrieving the spatial state of an object, and UR5 robot arms, for executing actions and exerting physical effects. In particular, the module takes the chance to discuss about representing and processing relevant data used in these systems. This helps to understand the API and argument descriptions with the mathematical implications in mind. 

## Motion Capture System (Mocap)
One of the most common requirements for `get_obs()` is about positioning interested objects in the environment. In the physical world, it is popular to use camera-based solutions to track objects without a physical contact. [Motion capture systems](https://en.wikipedia.org/wiki/Motion_capture) provide precise and rapid measurements by mounting several cameras whose field of views cover a designated area. The cameras receive rays from markers attached to the tracking objects. The rays can be actively emitted or passively reflected with certain wavelength such that they can be easily detected by the cameras. The markers form a cluster called _point cloud_ that contains an array of 3D locations. In general Mocap software systems can process the point cloud and identify subgroups that are attached to the same rigid object, and distribute the object _pose_ including position and orientation as a data stream. The position is simply a 3D vector with coordinates $[x, y, z]$. The orientation of a rigid body is more invovled and can be represented in different formats, such as [quaternion](https://en.wikipedia.org/wiki/Quaternion) i.e. $[x, y, z, w]$ in the Optitrack system. 



## Representing 3D Object Pose 
Let's dive a bit deeper into representing the 3D pose of a rigid object. We need at least 3 points that are not co-linear to reconstruct the location of all other points. These points are not necessarily within the rigid body volume: one can imagine fictitious parts extending infinitely in the space. With such a freedom, it is conventional to select 3 points forming a rectangular triangle with a unit length for each leg, see [](#rigidframe). This defines a rigid body frame to fully determine the pose of an object. Point $\mathbf{o}$ defines the rigid body position while the orientation can be described by two directions $\mathbf{r}_x = \mathbf{x} - \mathbf{o}$ and $\mathbf{r}_y = \mathbf{y} - \mathbf{o}$. With a bit redundancy, we can also obtain a third direction with $\mathbf{r}_z = \mathbf{r}_x \times \mathbf{r}_y$ to construct a reference frame. 

```{figure} ../images/rigidframe.png
---
name: rigidframe
---

A frame is used for representing the rigid body pose: position + orientation. The black dots are three points to form a rectangulalr triangle. The z-axis of the frame is formed as a cross product through the [right-hand rule](https://en.wikipedia.org/wiki/Right-hand_rule).
```

:::{note}
One can imagine a frame that is welded to the rigid body, or is _instaneuously coincident_ with the welded one. The latter is essentially _stationary_ and can find convenience in dynamics computation, see Chapter 3 & 8 of [@modernrobotics]. Roughly speaking, a welded frame takes a Lagrangian perspective like a first-person-view by riding on the welded rigid body particle. A stationary frame takes a Eulerian perspective like a third-person-view of the particles flowing into the anchored location, e.g. a fixed sensor measuring the pressure of a water flow. 
:::

Using notations with clearer semantics, the pose of a rigid body can be represented by a tuple of $(\mathbf{R}, \mathbf{p})$, with $\mathbf{p}$ representing the rigid body position as the position of the frame origin $\mathbf{o}$ and $\mathbf{R} = [\mathbf{r}_x, \mathbf{r}_y, \mathbf{r}_z]$ defining a $3 \times 3$ _rotation matrix_ for the orientation part. Again we emphasize that the choice of the reference point $\mathbf{o}$ is not unique. For intance, it is equally valid or sometimes preferred to define the frame at the centroid of the cuboid in [](#rigidframe). 



Rotation matrix $\mathbf{R} \in \mathbb{R}^{3 \times 3}$ is apparently a redundant parameterization of the rigid body orientation. We know that $\mathbf{r}_z$ is just derived from $\mathbf{r}_x$ and $\mathbf{r}_y$ so we can safely ignore this component in the $9$ parameters. $\mathbf{r}_x$ and $\mathbf{r}_y$ are also not arbitrary vectors, with their coordinates conforming to $\| \mathbf{r}_x \|_2 = 1$, $\| \mathbf{r}_y \|_2 = 1$ and $\mathbf{r}_x \cdot \mathbf{r}_y = 0$. These $3$ equations remove the freedom of $3$ extra parameters to choose their values, leaving the number of free orientation parameters to $9-3-3=3$. Intuitively, one can see this is true because the orientation of the cuboid in [](#rigidframe) can be attained by specifying the amount to rotate about each of the $3$ coordinate axes. [Euler angles](https://en.wikipedia.org/wiki/Euler_angles) is an example of using $3$ numbers for rigid body orientation. One may hear about one instance of that called "roll-yaw-pitch" convention that is intuitive and widely used in aerospace domain. 

Why a redundant representation like rotation matrix is still around when the intrinsic dimension of orientation is much more compact? The simple answer is using $3$ numbers to represent orientation will face difficulties in certain situations. This is rooted from the [geometry structure](https://en.wikipedia.org/wiki/Orthogonal_group#Special_orthogonal_group) about mathematical objects for describing rotations. In many literature and API references, one may find the usage of a $4$-number representation, such as the Mocap example above using [quaternion](https://en.wikipedia.org/wiki/Quaternion). A quaternion can be normalized i.e. $x^2 + y^2 + z^2 + w^2 = 1$ so the intrinsic dimension is still $3$. Another representation is [axis-angle](https://en.wikipedia.org/wiki/Axis–angle_representation) that reduces rotation to the effect of rotating about a direction $\mathbf{e}$ with a magnitude $\theta$, i.e. $(\mathbf{e}, \theta)$ with $\| \mathbf{e} \|_2 = 1$. Sometimes (e.g. for certain UR5 APIs) the direction and magnitude are blended as $\theta \mathbf{e}$ and one may recover the original form with a normalization.

There is nothing wrong to use one representation instead of another as long as it fits the need. In the end, they are just different appearances of the same underlying object. If the focus is solely about practice, it is probably more important to tell what representations those APIs are expecting and returning, and be aware of the conventions they are following, e.g. $[x, y, z, w]$ or $[w, x, y, z]$. Almost all libraries manipulating objects in a 3D world use one or several of these representations and provide utilities for [conversion between them](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.transform.Rotation.html). 

The plural of orientation representation implies that the time derivative of the orientation parameters cannot be taken as the angular velocity as it is. Otherwise we will end up with multiple angular velocities when the object rotates. To see how are the derivative and the actual angular velocity related, let's revisit the rotation matrix representation $\mathbf{R} = [\mathbf{r}_x, \mathbf{r}_y, \mathbf{r}_z]$. Note that the conditions for $\mathbf{r}_x$,  $\mathbf{r}_y$ and $\mathbf{r}_z$ can be concisely written as $\mathbf{R} \mathbf{R}^\top = \mathbf{I}^{3 \times 3}$, we can differentiate both sides and rearrange the equation:


$$
\label{eq-rotmatdifferential_deriv}
\begin{aligned}
& \dot{(\mathbf{R}\mathbf{R}^\top)} = \dot{\mathbf{R}}\mathbf{R}^\top + \mathbf{R}\dot{\mathbf{R}}^\top = 0 \\
\Rightarrow & \dot{\mathbf{R}}\mathbf{R}^\top = -\mathbf{R}\dot{\mathbf{R}}^{\top} = -(\dot{\mathbf{R}}\mathbf{R}^\top)^\top
\end{aligned}
$$

This means $\dot{\mathbf{R}}\mathbf{R}^\top$ is a [skew-symmetric](https://en.wikipedia.org/wiki/Skew-symmetric_matrix) matrix, with a form like:

$$
\label{eq-rotmatdifferential}
\begin{aligned}
\dot{\mathbf{R}}\mathbf{R}^\top = 
\begin{bmatrix}
0 & -w_z & w_y \\
w_z & 0 & -w_x \\
-w_y & w_x & 0
\end{bmatrix}
\end{aligned}
$$

Define $\mathbf{w} \times = \dot{\mathbf{R}}\mathbf{R}^\top$ with $\mathbf{w} = [w_x, w_y, w_z]^{\top}$. We see $\mathbf{w}$ is the 3-dimensional vector pinning down the angular velocity and the operation $\cdot \times$ converts the vector to the skew-symmetric matrix. Hence the matrix form of angular velocity is associated with the time-derivative of rotation matrix through a product with the rotation matrix itself. There are [similar product computation](https://ethz.ch/content/dam/ethz/special-interest/mavt/robotics-n-intelligent-systems/rsl-dam/documents/RobotDynamics2017/RD_HS2017script.pdf) for other orientation representations. The practical implication is that APIs often return a 6D array (3D for linear velocity and 3D for angular velocity) when a rigid body velocity is queried, and this must not be confused with a 6D array, e.g. when position and axis-angle are composed for the 3D pose of an object. 

:::{note}
Using $\mathbf{w} \times$ to convert $\mathbf{w}$ to a skew-symmetric matrix is semantically consistent with the [cross product](https://en.wikipedia.org/wiki/Cross_product) of two 3D vectors. In fact cross product can be understood as the matrix-vector product with the said skew-symmetric matrix, i.e. $\mathbf{w} \times \mathbf{v} = (\mathbf{w} \times) \cdot \mathbf{v}$, or vice-versa. 
:::

## UR5 Robot Arm - Joints, End-Effector and Controller

UR5 is a robot arm finding much popularity in industry and research. From a mechanism perspective, a robot arm is nothing more than a few rigid body links connected through articulations or say joints. Joints allow links to rotate relative to each other along one specific axis. This is called $1$ degree-of-freedom (DOF) hinge or revolute joint, for which only one value is needed to determine the pose of one connected link relative to the other. There are other types of joints e.g. [slider](https://en.wikipedia.org/wiki/Slider-crank_linkage), possibly with more values e.g. [ball joint](https://en.wikipedia.org/wiki/Ball_joint). We stick to 1-DOF hinge joint in what follows for its prevalent usage in robotics and character simulation. 

By fixing the robot basis, the poses of all links can be determined by a collection of joint angles, i.e. $\mathbf{q} = [q_1, q_2, q_3, q_4, q_5, q_6]$, see left in [](#ur5joint). This is called robot _configuration_ and all valid $\mathbf{q}$, when links are collision-free etc., comprises of a configuration space. The configuration space determines how flexible is the robot arm for realizing various postures. Apparently, more joints may yield more sophisticated configuration space and hence more complex motions. The number of DOFs is thus an important feature parameter for robot arms. 

```{figure} ../images/ur5joint.png
---
name: ur5joint
---
UR5 robot arm and its virtual counterpart. The arm "manipulates" objects by coordinating the motion of each individual joint and the mounted end-effector. 
```

UR5 is a 6-DOF robot arm, meaning it is possible to fully dictate the 6D pose of the last link. This is significant because arm manipulation is usually focused on the pose trajectory of a tool piece mounted on the last link. For instance, the tool can be a robot gripper for picking-and-place (right in [](#ur5joint)), a spray for painting, an extruder for 3D printing or even an anthropomorphic hand. These tools are called _end-effectors_ and the interested points on them for physical world interaction are tool center points (TCP). Representing a task with the end-effector state is obviously more intuitive for working with geometry in a Cartesian space. Literature or API manuals often use terms like Cartesian space, operational space and task space to clarify the situation, e.g. is an array of a length of 6 denoting a 6-DOF joint configuration or a 6-DOF end-effector pose. 

Robot arms are commonly actuated through electric motors mounted at joints, with sensors integrated for measuring the relative angles between the connected links. Joint position $\mathbf{q}$ and its change rate $\dot{\mathbf{q}}$ can thus be read and written in the RL loop of observation and action. When the robot links are deemed rigid it is also possible to have a fairly accurate estimate of the operational space state through _forward kinematics_ (see below). Most APIs provide calls like `getTCPPose()` and `getTCPSpeed()` to do this. 

Actions for joint position or velocity are actually a desired signal sent to low-level robot control, because a robot can not teleport between position states in real-world. Some robots also expose interface for commanding joint forces so RL can learn a forceful policy for some skillful tasks, e.g. assembling tightly mating parts. Similarly, there are APIs to write desired operational space state while care must be taken since the path of moving towards the target values may not always be feasible. 


:::{note}
Robots with less or more than 6 joints have tasks they are specialized on. [Structures](https://en.wikipedia.org/wiki/SCARA) with 3 or 4 DOFs are sometimes favoured for having less motor costs and improved structural rigidity. With more than 6 DOFs, robot arms enjoy redundancy for realizing the same end-effector pose. This sometimes can be beneficial, e.g. to avoid environment obstacles. 
:::



## Robot Arm Kinematics
The mathematical description between the link pose and joint configuration position $\mathbf{q}$ is central to using $\mathbf{q}$ to realize desired end-effector poses. This mathematical relation is called robot arm _kinematics_. Taking the UR5 example, we can denote each link pose with a rigid body frame (remember we can choose whatever convenient point as the origin). A link frame will be displaced with respect to its parent link under different joint position $\mathbf{q}$, see the video clip below. 

In [47]:
import mujoco
import mediapy as media
import numpy as np

scene_spec = mujoco.MjSpec.from_file("../data/mujoco/scene.xml")
rg2_spec = mujoco.MjSpec.from_file("../data/mujoco/onrobot_rg2.xml")
#mount rg2 to ur5e attachment_site
# attachment_site = next(s for s in scene_spec.sites if s.name == "attachment_site")
# attachment_site.attach(rg2_spec.worldbody, "gripper_base", "")

mjc_model = scene_spec.compile()
mjc_data = mujoco.MjData(mjc_model)
mujoco.mj_resetData(mjc_model, mjc_data) # reset

renderer = mujoco.Renderer(mjc_model, height=480, width=640)
scene_option = mujoco.MjvOption()
scene_option.frame = mujoco.mjtFrame.mjFRAME_BODY
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = True

frames = []

for t in np.linspace(0, 2*np.pi, 100):
    mjc_data.qpos = np.deg2rad([-90 - 20*np.sin(t), -60-20*np.sin(t), 90+20*np.sin(t), 0, 0, 0]) #arm and hand joints
    mujoco.mj_step(mjc_model, mjc_data)
    renderer.update_scene(mjc_data, scene_option=scene_option)
    pixels = renderer.render()
    frames.append(pixels)


media.show_video(frames, fps=10)

renderer.close()

This browser does not support the video tag.


Concretely, the link pose defines transformation that changes the coordinate of any point on the rigid body, see [](#prevsuccframes). The spatial position $\mathbf{p}$ can have a coordinate representation with reference to either link $i$ or its parent link $i-1$. We can use right subscript to denote the object that the quantity is measured with reference to, e.g. $\mathbf{p}_{i-1, i}$ stands the position of (the origin of) link frame $i$ measured relative to (the origin of) link frame $i-1$. The left upperscript denotes the frame that is used as the basis to express the coordinate. In many cases, the reference frame is the same one used for the expression basis so one of them can be omitted, e.g. ${}^{i-1}\mathbf{p}_i$ or $\mathbf{p}_{i-1, i}$. With these in mind, one can write following relation:

$$
\label{eq-coordtrans}
\begin{aligned}
{}^{i-1}\mathbf{p}_{i-1, i} & = {}^{i-1}\mathbf{R}_i {}^{i}\mathbf{p}_{i-1, i}    \\
{}^{i-1}\mathbf{p}_{i-1, c} & = {}^{i-1}\mathbf{p}_{i-1, i} + {}^{i-1}\mathbf{p}_{i, c} = {}^{i-1}\mathbf{R}_i {}^{i}\mathbf{p}_{i, c} + {}^{i-1}\mathbf{p}_{i-1, i}
\end{aligned}
$$
with $\mathbf{R}$ as the result of rotating along the $x$-axis (the dashed frame):

$$
\label{eq-elemrot}
{}^{i-1}\mathbf{R}_i = 
\begin{bmatrix}
1 & 0   & 0 \\
0 & \cos q_i & -\sin q_i    \\
0 & \sin q_i & \cos q_i
\end{bmatrix} 
$$

```{figure} ../images/prevsuccframes.png
---
name: prevsuccframes
---

Point representation in the frame of link $i$, i.e. ${}^{i}\mathbf{p}$, is associated to its counterpart in the previous link frame, i.e. ${}^{i-1}\mathbf{p}$, via the relative angle i.e. the $i$-th joint position $q_{i-1, i}$ or in short $q_i$. The coordinate of any point $c$ on the successor link $i$ can be transformed to a coordinate measured and expressed in the parent link $i-1$.
```

A compact way of writing [](#eq-coordtrans) is to construct _homogenous transformation matrix_ with $(\mathbf{R}, \mathbf{p})$:

$$
\label{eq-rigidtrans}
{}^{i-1}\mathbf{T}_i(q_i) = 
\begin{bmatrix}
{}^{i-1}\mathbf{R}_i   & {}^{i-1}\mathbf{p}_i \\
\mathbf{0} &  1   \\
\end{bmatrix}
\\ 
{}^{i-1}\mathbf{p}_c =  
\begin{bmatrix}
{}^{i-1}\mathbf{p}_{i-1, c} \\
1   \\
\end{bmatrix} = {}^{i-1}\mathbf{T}_i \cdot 
\begin{bmatrix}
{}^{i}\mathbf{p}_{i, c} \\
1   \\
\end{bmatrix} = 
{}^{i-1}\mathbf{T}_i \cdot {}^{i}\mathbf{p}_c
$$

${}^{i-1}\mathbf{T}_i$ can thus be used to represent the pose of link $i$ with a reference to link $i-1$. It is a function of joint position $q_i$ since ${}^{i}\mathbf{p}_{i, c}$ and ${}^{i}\mathbf{p}_{i-1, i}$ are constant as long as link $i$ does not deform. One can intuitively see between which reference frames the transformation is happening by cancelling out adjacent subscripts: ${}^{i-1}\mathbf{p}_{c} = {}^{i-1}\mathbf{T}_{\cancel{i}} \cdot {}^{\cancel{i}}\mathbf{p}_c$. An immediate result from this is that forward kinematics, which associate link poses to joint position $\mathbf{q}$, can be obtained by cascading transformation matrices between arm links. For instance, the pose of the last link of UR5 with reference to the basis link $0$ is:

$$
\label{eq-urfk}
{}^{0}\mathbf{T}_{6} = \text{FK}(\mathbf{q}) = {}^{0}\mathbf{T}_{1}(q_1) \cdot {}^{1}\mathbf{T}_{2}(q_2) \cdot {}^{2}\mathbf{T}_{3}(q_3) \cdot {}^{3}\mathbf{T}_{4}(q_4) \cdot {}^{4}\mathbf{T}_{5}(q_5) \cdot {}^{5}\mathbf{T}_{6}(q_6)
$$

when the link $0$ is fixed with a known pose with reference to the world coordinate system, one can estimate the last link pose as well as the position of any point such as TCP in the spatial space given the reading of $\mathbf{q}$. Forward kinematics hence boil down to a sequence of matrix multiplication. 

With forward kinematics to get end-effector pose from joint position, it is legit to ask how to do a reverse process a.k.a inverse kinematics. After all, it is much more intuitive to design a serie of tool poses and then derive the joint positions to realize them, e.g. through $\mathbf{q} = \text{InvKin}({}^{0}\mathbf{T}_{6})$. It is also common in cases like animating characters where a designer might prefer to edit the hand or foot pose directly. Inverse kinematics are usually not analytically available. Exceptions exist when joints are arranged in some special ways. General approaches rely on numerical algorithms to solve the equation of [](#eq-urfk). The algorithms often rely on the differential of forward kinematics [](#eq-urfk), which captures the displacement of the interested link caused by the perturbation to $\mathbf{q}$:

$$
\label{eq-diffkin}
\delta \mathbf{T} = \mathbf{J}(\mathbf{q}) \delta \mathbf{q}
$$

Starting with an initial guess $\mathbf{q}^{0}$ as the $\text{InvKin}$ solution, one can iteratively refine this guess by solving [](#eq-diffkin) and update with e.g. $\mathbf{q}^{k+1} = \mathbf{q}^{k} + \delta \mathbf{q} = \mathbf{q}^{k} + \alpha \mathbf{J}^{-1}(\mathbf{q}^k) \delta \mathbf{T}^k$. $\delta \mathbf{T}$ can be written as a 6D vector, 3 for translational and 3 for rotational displacements, since we know from [](#eq-rotmatdifferential) that infinitesimal rotation is associated to angular velocity defined by 3 values:

$$
\label{eq-geomjacobian}
\begin{bmatrix}
\mathbf{v}  \\
\mathbf{\omega}
\end{bmatrix} = \mathbf{J}(\mathbf{q}) \dot{\mathbf{q}}
$$
with $\mathbf{v}$ and $\mathbf{\omega}$ denoting the linear and angular velocities of the interested link. $\mathbf{J}(\cdot)$ is hence a Jacobian matrix with 6 rows and a column number same as the number of total joint DOFs, e.g. for UR5 $\mathbf{J} \in \mathbb{R}^{6 \times 6}$. Note that this is _geometry_ Jacobian which is different from analytically differentiating [](#eq-urfk). This is often provided as model APIs because associating the underlying link velocity looks more fundamental, independent of the adopted representations (euler angle, quaternion, etc.) for the rotational part. Of course, similar to the case in [](#eq-rotmatdifferential_deriv), the analytical Jacobian can be associated with a simple matrix multiplication.

Let's try this $\text{InvKin}$ idea in the [MuJoCo](https://github.com/google-deepmind/mujoco) simulator which provides APIs related to these kinematics calculation. We can derive a sequence of $\mathbf{q}$ that will make the end-effector follow a straight line. 

In [170]:
import mujoco
import numpy as np
import copy 

def ik_pseudo_inverse(model, data, link_name, target_pos, target_rot, init_guess, num_itrs=20, regularization=0.0):
    #make a copy of data so we wont mess up the one used by simulation
    data_copy = copy.copy(data)
    data_copy.qpos = init_guess
    for i in range(num_itrs):
        mujoco.mj_kinematics(model, data_copy)
        #get the last link pose
        pos = data_copy.body(link_name).xpos
        rot = data_copy.body(link_name).xquat

        #get error of the pose, note the orientation part: quat_res = quat_a * neg(quat_b) similar to res = a - b in the euclidean case
        #the infinitesimal is 3d vector 
        err_pos = target_pos - pos
        err_rot = np.empty(3) 
        mujoco.mju_subQuat(err_rot, target_rot, rot)
        err = np.concatenate([err_pos, err_rot]) * 0.01

        #get jacobian and do the assembly
        jacp, jacr = np.empty((3, model.nv)), np.empty((3, model.nv))
        mujoco.mj_jac(model, data_copy, jacp, jacr, pos, data_copy.body(link_name).id)
        jac = np.concatenate([jacp, jacr])

        # dp = Jac(q) * dq --> J^T * dp = J^T * J * dq --> dq = (J^T*J + \lambda * I)^{-1} * dp
        hess_approx = jac.T.dot(jac)
        delta = jac.T.dot(err)
        if regularization > 0:
            hess_approx += np.eye(hess_approx.shape[0]) * regularization
            dq = np.linalg.solve(hess_approx, delta)
        else:
            dq = np.linalg.lstsq(hess_approx, delta, rcond=-1)[0]

        #strictly speaking here needs mj_integratePos for general joint types
        data_copy.qpos += dq
    return data_copy.qpos

#create a scene with UR5 robot
scene_spec = mujoco.MjSpec.from_file("../data/mujoco/scene.xml")
mjc_model = scene_spec.compile()
mjc_data = mujoco.MjData(mjc_model)
mujoco.mj_resetData(mjc_model, mjc_data) # reset

renderer = mujoco.Renderer(mjc_model, height=480, width=640)
scene_option = mujoco.MjvOption()

#drive UR5 along a straight line with a fixed orientation
init_jnt_pos = np.deg2rad([-90, -60, 90, -30, 0, 0])
mjc_data.qpos = init_jnt_pos
frames = []

#get the last link pose, note the link name defined in ur5e.xml
mujoco.mj_kinematics(mjc_model, mjc_data)
pos = mjc_data.body('wrist_3_link').xpos.copy()
rot = mjc_data.body('wrist_3_link').xquat.copy()

#do a linear interpolation for the position component
pos_traj = np.linspace(pos, pos + np.array([0, 0, 0.2]), 20)
for waypnt in pos_traj:
    init_guess = mjc_data.qpos
    desired_qpos = ik_pseudo_inverse(mjc_model, mjc_data, 'wrist_3_link', waypnt, rot, init_guess, num_itrs=20)

    mjc_data.qpos = desired_qpos
    mujoco.mj_step(mjc_model, mjc_data)
    renderer.update_scene(mjc_data, scene_option=scene_option)
    pixels = renderer.render()
    frames.append(pixels)

media.show_video(frames, fps=10)

renderer.close()

This browser does not support the video tag.


Note that it is fine to use `jacp` as a $3\times 6$ Jacobian if $\text{InvKin}$ only concerns the end-effector position. The realized orientation cannot be guaranteed to be a fixed value because the algorithms now have the freedom to use all 6 joints to fulfill 3 values of the end-effector position. Moreover, the inversion of Jacobian implies $\text{InvKin}$ may fail when $\det(\mathbf{J}) = 0$. The $\mathbf{q}$- _dependent_ linear relation as in [](#eq-geomjacobian) allows for using linear algebra tools to analyze operational space motion that can be induced by joint motion. When $\mathbf{q}$ causes a singular $\mathbf{J}$ (called configuration singularity) or close to that, a small operational space velocity will require extremely large joint velocity which can exceed the hardware capacity. Imagine the configuration of an UR5 arm stretching straight up, it will be impossible to generate velocity along the straight arm. Thus it is necessary to be aware of the feasibility of an end-effector trajectory for the robot to track. Numerically, the unstability due to inverting a singular matrix can be alleviated by adding a regularized term, see the usage of `regularization` in the code above. This may however compromise the solution accuracy. Other popular approaches solve $\text{InvKin}$ with the Jacobian transpose in an optimization formulation, with the possibility of incorporating joint position limits. 